# SQL Guardrails and Validation

A text-to-SQL agent that **generates** plausible SQL is not production-ready until every query passes **guardrails** before execution. LLMs can emit `DROP TABLE`, unbounded scans, queries against forbidden tables, or injected fragments — even with good schema grounding.

**Guardrails** are layered checks between SQL generation and the database:

```
NL question → SQL generation → GUARDRAILS → database
                              │
                    ┌─────────┼─────────┐
                    ▼         ▼         ▼
               statement   schema    resource
               allowlist   allowlist  limits
```

This notebook builds **ShopCo Guarded SQL** — a validation pipeline for support-agent queries:

1. **Statement** guards — SELECT-only, no DDL/DML
2. **Schema** guards — allowed tables/columns per role
3. **Resource** guards — `LIMIT`, max rows, timeout simulation
4. **Injection** guards — suspicious patterns blocked
5. **Parameterized** execution path for user-supplied values
6. **Audit log** — every allow/block decision recorded
7. Integration with a **grounded text-to-SQL** agent loop

Self-contained SQLite + plain Python. No CLI required.

In [ ]:
"""
ShopCo SQL Guardrails — validation lab.
"""

import json
import os
import re
import sqlite3
import time
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def create_shopco_db() -> sqlite3.Connection:
    conn = sqlite3.connect(":memory:")
    conn.row_factory = sqlite3.Row
    conn.executescript("""
        CREATE TABLE customers (
            customer_id INTEGER PRIMARY KEY,
            email TEXT NOT NULL,
            name TEXT NOT NULL,
            tier TEXT DEFAULT 'standard',
            ssn_last4 TEXT
        );
        CREATE TABLE orders (
            order_id TEXT PRIMARY KEY,
            customer_id INTEGER NOT NULL,
            status TEXT NOT NULL,
            total_usd REAL NOT NULL,
            tracking TEXT,
            created_at TEXT NOT NULL
        );
        CREATE TABLE products (
            product_id INTEGER PRIMARY KEY,
            sku TEXT NOT NULL,
            name TEXT NOT NULL,
            price_usd REAL NOT NULL
        );
        CREATE TABLE internal_payroll (
            employee_id INTEGER PRIMARY KEY,
            salary_usd REAL NOT NULL,
            department TEXT
        );
    """)
    conn.executemany("INSERT INTO customers VALUES (?,?,?,?,?)", [
        (1, "alice@shopco.com", "Alice Nguyen", "gold", "1234"),
        (2, "bob@shopco.com", "Bob Smith", "standard", "5678"),
    ])
    conn.executemany("INSERT INTO orders VALUES (?,?,?,?,?,?)", [
        ("ORD-1001", 1, "shipped", 89.50, "1Z999AA10123456784", "2026-07-01"),
        ("ORD-1002", 2, "processing", 42.00, None, "2026-07-08"),
    ])
    conn.executemany("INSERT INTO products VALUES (?,?,?,?)", [
        (1, "MOUSE-01", "Wireless Mouse", 29.99),
        (2, "HUB-02", "USB-C Hub", 49.99),
    ])
    conn.executemany("INSERT INTO internal_payroll VALUES (?,?,?)", [
        (101, 95000.0, "engineering"), (102, 72000.0, "support"),
    ])
    conn.commit()
    return conn


DB = create_shopco_db()
print("ShopCo DB with sensitive tables ready")

---

## 1. Defense in Depth

| Layer | Blocks | Example |
|-------|--------|--------|
| **Statement** | Non-SELECT, multi-statement | `DROP TABLE orders` |
| **Schema** | Forbidden tables/columns | `SELECT salary FROM internal_payroll` |
| **Resource** | Unbounded scans | `SELECT * FROM orders` (no LIMIT) |
| **Injection** | Comment/union tricks | `ORD-1001' OR '1'='1` |
| **Role** | Scope per agent persona | Support cannot read `ssn_last4` |
| **Audit** | Silent failures | Log every decision with reason |

---

## 2. Role-Based Schema Policy

Different agents get different **table and column allowlists**.

In [ ]:
@dataclass
class SQLRolePolicy:
    role: str
    allowed_tables: set[str]
    denied_columns: dict[str, set[str]]  # table → blocked columns
    max_rows: int = 100
    statement_allowlist: set[str] = field(default_factory=lambda: {"SELECT"})


SUPPORT_POLICY = SQLRolePolicy(
    role="support_agent",
    allowed_tables={"customers", "orders", "products"},
    denied_columns={"customers": {"ssn_last4"}},
    max_rows=50,
)

ANALYST_POLICY = SQLRolePolicy(
    role="analyst_agent",
    allowed_tables={"customers", "orders", "products"},
    denied_columns={},
    max_rows=500,
)

print(f"Support may query: {SUPPORT_POLICY.allowed_tables}")
print(f"Denied columns: {SUPPORT_POLICY.denied_columns}")

---

## 3. Validation Result Model

In [ ]:
class GuardVerdict(str, Enum):
    ALLOW = "allow"
    BLOCK = "block"
    REWRITE = "rewrite"


@dataclass
class ValidationIssue:
    code: str
    message: str
    severity: str = "error"


@dataclass
class SQLValidationResult:
    verdict: GuardVerdict
    original_sql: str
    sanitized_sql: str = ""
    issues: list[ValidationIssue] = field(default_factory=list)

    @property
    def ok(self) -> bool:
        return self.verdict in {GuardVerdict.ALLOW, GuardVerdict.REWRITE}

---

## 4. Layer 1 — Statement Allowlist

In [ ]:
FORBIDDEN_STMT = re.compile(
    r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|CREATE|REPLACE|TRUNCATE|ATTACH|DETACH|PRAGMA|GRANT|REVOKE)\b",
    re.IGNORECASE,
)
MULTI_STMT = re.compile(r";\s*\S")


def check_statement(sql: str, policy: SQLRolePolicy) -> list[ValidationIssue]:
    issues: list[ValidationIssue] = []
    cleaned = sql.strip()
    if not cleaned:
        issues.append(ValidationIssue("EMPTY", "empty SQL"))
        return issues
    if MULTI_STMT.search(cleaned):
        issues.append(ValidationIssue("MULTI_STMT", "multiple statements not allowed"))
    if FORBIDDEN_STMT.search(cleaned):
        issues.append(ValidationIssue("FORBIDDEN_STMT", "only SELECT permitted"))
    first = cleaned.split()[0].upper() if cleaned.split() else ""
    if first not in policy.statement_allowlist:
        issues.append(ValidationIssue("STMT_NOT_ALLOWED", f"statement {first} not in allowlist"))
    return issues


for sql in [
    "SELECT order_id FROM orders",
    "DELETE FROM orders WHERE order_id = 'ORD-1001'",
    "SELECT 1; DROP TABLE orders",
]:
    print(sql, "→", [i.code for i in check_statement(sql, SUPPORT_POLICY)])

---

## 5. Layer 2 — Schema Allowlist (Tables & Columns)

In [ ]:
ALL_TABLES = {"customers", "orders", "products", "internal_payroll"}
ALL_COLUMNS = {
    "customers": {"customer_id", "email", "name", "tier", "ssn_last4"},
    "orders": {"order_id", "customer_id", "status", "total_usd", "tracking", "created_at"},
    "products": {"product_id", "sku", "name", "price_usd"},
    "internal_payroll": {"employee_id", "salary_usd", "department"},
}


def extract_tables(sql: str) -> set[str]:
    found = set()
    for m in re.finditer(r"\bFROM\s+([a-zA-Z_][a-zA-Z0-9_]*)", sql, re.I):
        found.add(m.group(1).lower())
    for m in re.finditer(r"\bJOIN\s+([a-zA-Z_][a-zA-Z0-9_]*)", sql, re.I):
        found.add(m.group(1).lower())
    return found


def extract_selected_columns(sql: str) -> set[str]:
    m = re.search(r"SELECT\s+(.+?)\s+FROM", sql, re.I | re.S)
    if not m:
        return set()
    clause = m.group(1)
    if clause.strip() == "*":
        return {"*"}
    cols = set()
    for part in clause.split(","):
        part = part.strip()
        col = part.split()[-1].split(".")[-1].lower()
        cols.add(col)
    return cols


def check_schema(sql: str, policy: SQLRolePolicy) -> list[ValidationIssue]:
    issues: list[ValidationIssue] = []
    tables = extract_tables(sql)
    for t in tables:
        if t not in policy.allowed_tables:
            issues.append(ValidationIssue("TABLE_DENIED", f"table {t} not allowed for {policy.role}"))
    cols = extract_selected_columns(sql)
    if "*" in cols:
        for t in tables:
            denied = policy.denied_columns.get(t, set())
            if denied:
                issues.append(ValidationIssue("STAR_DENIED", f"SELECT * blocked on {t} due to sensitive columns {denied}"))
    else:
        for t in tables:
            denied = policy.denied_columns.get(t, set())
            for col in cols:
                if col in denied:
                    issues.append(ValidationIssue("COLUMN_DENIED", f"column {t}.{col} denied for {policy.role}"))
    return issues


print(check_schema("SELECT * FROM orders", SUPPORT_POLICY))
print(check_schema("SELECT ssn_last4 FROM customers", SUPPORT_POLICY))
print(check_schema("SELECT salary_usd FROM internal_payroll", SUPPORT_POLICY))

---

## 6. Layer 3 — Resource Limits

In [ ]:
def enforce_limit(sql: str, max_rows: int) -> tuple[str, list[ValidationIssue]]:
    issues: list[ValidationIssue] = []
    cleaned = sql.strip().rstrip(";")
    if not re.search(r"\bLIMIT\s+\d+", cleaned, re.I):
        cleaned += f" LIMIT {max_rows}"
        issues.append(ValidationIssue("LIMIT_ADDED", f"appended LIMIT {max_rows}", severity="info"))
    else:
        m = re.search(r"\bLIMIT\s+(\d+)", cleaned, re.I)
        if m and int(m.group(1)) > max_rows:
            cleaned = re.sub(r"LIMIT\s+\d+", f"LIMIT {max_rows}", cleaned, flags=re.I)
            issues.append(ValidationIssue("LIMIT_CAPPED", f"capped LIMIT to {max_rows}", severity="info"))
    return cleaned, issues


sql, issues = enforce_limit("SELECT order_id FROM orders", 50)
print(sql, issues)

---

## 7. Layer 4 — Injection Pattern Detection

In [ ]:
INJECTION_PATTERNS = [
    (r"--", "SQL comment"),
    (r"/\*", "block comment"),
    (r"\bUNION\s+SELECT", "UNION injection"),
    (r"';\s*", "quote termination"),
    (r"\bOR\s+['\"]?1['\"]?\s*=\s*['\"]?1", "always-true OR"),
    (r"\bSLEEP\s*\(", "timing attack"),
]


def check_injection(sql: str) -> list[ValidationIssue]:
    issues: list[ValidationIssue] = []
    for pattern, label in INJECTION_PATTERNS:
        if re.search(pattern, sql, re.I):
            issues.append(ValidationIssue("INJECTION", f"suspicious pattern: {label}"))
    return issues


malicious = "SELECT order_id FROM orders WHERE order_id = 'ORD-1001' OR '1'='1'"
print(check_injection(malicious))

---

## 8. Parameterized Queries — Safe User Values

Never interpolate user input into SQL strings. Use bound parameters.

In [ ]:
def lookup_order_safe(conn: sqlite3.Connection, order_id: str) -> list[dict[str, Any]]:
    if not re.match(r"^ORD-[0-9]{4}$", order_id):
        raise ValueError("invalid order_id format")
    rows = conn.execute(
        "SELECT order_id, status, total_usd, tracking FROM orders WHERE order_id = ? LIMIT 10",
        (order_id,),
    ).fetchall()
    return [dict(r) for r in rows]


print(lookup_order_safe(DB, "ORD-1001"))
try:
    lookup_order_safe(DB, "ORD-1001' OR '1'='1")
except ValueError as e:
    print("Blocked bad input:", e)

---

## 9. Unified Validation Pipeline

In [ ]:
SQL_AUDIT: list[dict[str, Any]] = []


def validate_sql(sql: str, policy: SQLRolePolicy) -> SQLValidationResult:
    issues: list[ValidationIssue] = []
    issues.extend(check_statement(sql, policy))
    issues.extend(check_injection(sql))
    if not any(i.severity == "error" for i in issues):
        issues.extend(check_schema(sql, policy))

    errors = [i for i in issues if i.severity == "error"]
    if errors:
        result = SQLValidationResult(GuardVerdict.BLOCK, sql, issues=issues)
    else:
        sanitized, limit_issues = enforce_limit(sql, policy.max_rows)
        issues.extend(limit_issues)
        verdict = GuardVerdict.REWRITE if limit_issues else GuardVerdict.ALLOW
        result = SQLValidationResult(verdict, sql, sanitized_sql=sanitized, issues=issues)

    SQL_AUDIT.append({
        "ts": utc_now(), "role": policy.role, "verdict": result.verdict.value,
        "sql": sql[:120], "issues": [i.code for i in result.issues],
    })
    return result


TEST_SQL = [
    ("SELECT order_id, status FROM orders WHERE order_id = 'ORD-1001'", "valid"),
    ("DELETE FROM orders", "destructive"),
    ("SELECT salary_usd FROM internal_payroll", "forbidden table"),
    ("SELECT ssn_last4 FROM customers", "PII column"),
    ("SELECT order_id FROM orders WHERE order_id = 'X' OR '1'='1'", "injection"),
]

for sql, label in TEST_SQL:
    r = validate_sql(sql, SUPPORT_POLICY)
    print(f"[{label}] {r.verdict.value} — {[i.code for i in r.issues if i.severity=='error']}")

---

## 10. Guarded Executor

In [ ]:
@dataclass
class ExecutionResult:
    sql: str
    rows: list[dict[str, Any]]
    row_count: int
    latency_ms: float
    blocked: bool = False
    block_reason: str = ""


class GuardedSQLExecutor:
    def __init__(self, conn: sqlite3.Connection, policy: SQLRolePolicy) -> None:
        self.conn = conn
        self.policy = policy

    def execute(self, sql: str) -> ExecutionResult:
        validation = validate_sql(sql, self.policy)
        if not validation.ok:
            reason = "; ".join(i.message for i in validation.issues if i.severity == "error")
            return ExecutionResult(sql=sql, rows=[], row_count=0, latency_ms=0, blocked=True, block_reason=reason)
        to_run = validation.sanitized_sql or sql
        start = time.perf_counter()
        try:
            rows = [dict(r) for r in self.conn.execute(to_run).fetchall()]
            ms = (time.perf_counter() - start) * 1000
            return ExecutionResult(sql=to_run, rows=rows, row_count=len(rows), latency_ms=ms)
        except sqlite3.Error as exc:
            return ExecutionResult(sql=to_run, rows=[], row_count=0, latency_ms=0, blocked=True, block_reason=str(exc))


executor = GuardedSQLExecutor(DB, SUPPORT_POLICY)

good = executor.execute("SELECT order_id, status FROM orders")
bad = executor.execute("SELECT * FROM internal_payroll")
print("Good:", good.row_count, "rows", f"({good.latency_ms:.2f}ms)")
print("Blocked:", bad.blocked, bad.block_reason)

---

## 11. Offline NL→SQL + Guardrails Agent

In [ ]:
def offline_plan_sql(question: str) -> str | None:
    q = question.lower()
    m = re.search(r"ORD-[0-9]{4}", question.upper())
    if m:
        return f"SELECT order_id, status, total_usd FROM orders WHERE order_id = '{m.group(0)}'"
    if "how many" in q and "order" in q:
        return "SELECT COUNT(*) AS cnt FROM orders"
    if "payroll" in q or "salary" in q:
        return "SELECT employee_id, salary_usd FROM internal_payroll"
    if "ssn" in q:
        return "SELECT name, ssn_last4 FROM customers"
    return None


@dataclass
class GuardedAgentResult:
    question: str
    planned_sql: str | None
    execution: ExecutionResult | None
    answer: str
    trace: list[str] = field(default_factory=list)


class GuardedTextToSQLAgent:
    def __init__(self, executor: GuardedSQLExecutor) -> None:
        self.executor = executor

    def run(self, question: str) -> GuardedAgentResult:
        trace: list[str] = []
        sql = offline_plan_sql(question)
        if not sql:
            return GuardedAgentResult(question, None, None, "Could not plan SQL.", ["plan:fail"])
        trace.append(f"plan:{sql[:50]}")
        result = self.executor.execute(sql)
        trace.append(f"execute:blocked={result.blocked}")
        if result.blocked:
            answer = f"Query blocked by guardrails: {result.block_reason}"
        elif not result.rows:
            answer = "No rows returned."
        else:
            answer = f"{pretty(result.rows[:3])}\n[SQL] {result.sql}"
        return GuardedAgentResult(question, sql, result, answer, trace)


agent = GuardedTextToSQLAgent(executor)

for q in [
    "Status of ORD-1001",
    "Show employee salaries",
    "List customer SSN numbers",
    "How many orders?",
]:
    r = agent.run(q)
    print(f"\nQ: {q}")
    print(f"  {r.answer[:90]}...")

---

## 12. Human-in-the-Loop for High-Risk Queries

Some queries pass syntax checks but need **human approval** — e.g., large exports or PII-adjacent fields for analysts.

In [ ]:
HITL_REQUIRED_CODES = {"COLUMN_DENIED", "TABLE_DENIED", "STAR_DENIED"}


def needs_human_approval(validation: SQLValidationResult) -> bool:
    return any(i.code in HITL_REQUIRED_CODES for i in validation.issues if i.severity == "error")


risky = validate_sql("SELECT email FROM customers", SUPPORT_POLICY)
print("Needs HITL:", needs_human_approval(risky), risky.issues)

---

## 13. Audit Log Review

In [ ]:
print(f"Audit entries: {len(SQL_AUDIT)}")
blocks = [e for e in SQL_AUDIT if e["verdict"] == "block"]
print(f"Blocked: {len(blocks)}")
for e in SQL_AUDIT[-5:]:
    print(f"  [{e['verdict']}] {e['issues']} — {e['sql'][:60]}...")

---

## 14. Guardrail Test Suite

In [ ]:
GUARDRAIL_TESTS: list[dict[str, Any]] = [
    {"sql": "SELECT order_id FROM orders", "expect": "allow"},
    {"sql": "DROP TABLE orders", "expect": "block"},
    {"sql": "SELECT * FROM internal_payroll", "expect": "block"},
    {"sql": "SELECT ssn_last4 FROM customers", "expect": "block"},
    {"sql": "SELECT order_id FROM orders; DELETE FROM orders", "expect": "block"},
    {"sql": "SELECT order_id FROM orders WHERE id = 'x' OR '1'='1'", "expect": "block"},
]

passed = 0
for t in GUARDRAIL_TESTS:
    v = validate_sql(t["sql"], SUPPORT_POLICY)
    got = "block" if v.verdict == GuardVerdict.BLOCK else "allow"
    ok = got == t["expect"] or (t["expect"] == "allow" and v.verdict == GuardVerdict.REWRITE)
    passed += int(ok)
    mark = "✓" if ok else "✗"
    print(f"{mark} expect={t['expect']} got={v.verdict.value} | {t['sql'][:50]}")
print(f"\nGuardrail tests: {passed}/{len(GUARDRAIL_TESTS)}")

---

## 15. Common Mistakes

| Mistake | Risk | Fix |
|---------|------|-----|
| Trust LLM SQL blindly | Data loss / leak | Multi-layer validation |
| String interpolation for IDs | SQL injection | Parameterized queries |
| No row LIMIT | Memory/DoS | Auto-append LIMIT |
| Same policy for all roles | PII exposure | Role-based allowlists |
| No audit log | Compliance gap | Log verdict + SQL hash |
| Validation only at dev time | Production bypass | Validate at every execution |

---

## 16. Production Checklist

- [ ] SELECT-only default for support agents
- [ ] Table/column allowlists per role
- [ ] Auto LIMIT + max row cap
- [ ] Injection pattern detection
- [ ] Parameterized queries for user literals
- [ ] Audit log with verdict, role, latency
- [ ] HITL gate for sensitive query classes
- [ ] Automated guardrail regression tests in CI

---

## 17. Optional Live LLM with Guardrails

Generate SQL with an LLM, then **always** pass through `validate_sql` before `GuardedSQLExecutor`.

In [ ]:
def live_then_guard(question: str) -> GuardedAgentResult:
    if not USE_LIVE_LLM:
        return agent.run(question)
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    schema = "Tables: customers, orders, products. SELECT only. No PII."
    sql = llm.invoke(f"{schema}\nQuestion: {question}\nSQL:").content.strip()
    result = executor.execute(sql)
    answer = result.block_reason if result.blocked else pretty(result.rows)
    return GuardedAgentResult(question, sql, result, answer, ["live:llm"])


demo = live_then_guard("How many orders?")
print(demo.answer[:120])

---

## 18. Quiz

1. Name four guardrail layers between SQL generation and execution.
2. Why use parameterized queries instead of f-strings for order IDs?
3. What should happen when `SELECT *` would expose denied columns?
4. When is `GuardVerdict.REWRITE` appropriate?
5. Why log blocked queries, not just successful ones?

---

## 19. Summary

**SQL guardrails** turn text-to-SQL from a demo into a controlled interface. The ShopCo pipeline validates:

1. **Statement** — SELECT-only, no multi-statement
2. **Schema** — role-based table/column allowlists
3. **Resources** — LIMIT enforcement
4. **Injection** — suspicious pattern blocking
5. **Execution** — `GuardedSQLExecutor` + audit log

Schema grounding (previous notebook) ensures correct identifiers; **guardrails** ensure safe execution. Next: **combining RAG and SQL agents** in one support workflow.